In [2]:
from pandas import DataFrame
from abc import ABC, abstractmethod
from typing import List, TypeVar, Generic, Tuple
from transformers import AutoTokenizer, BatchEncoding
from tqdm import tqdm

BAIT_WORDS = [
    "ALERT(Ă|A)", "alerta", "Alertă",
    "ANALIZA", "analiza", "Analiza",
    "Ancheta", "ANCHETA", "ancheta",
    "BREAKING NEWS", "Breaking News", "breaking news",
    "CAPTURA", "captura", "Captura",
    "CLIP", "clip", "Clip",
    "CONFLICT", "conflict", "Conflict",
    "CONFRUNTARE", "confruntare", "Confruntare",
    "CONTROVERSAT", "controversat", "Controversat",
    "CRIZ(Ă|A)", "criză", "Criză",
    "DECLARAȚIE", "declarație", "Declarație",
    "DEZASTU", "dezastru", "Dezastru",
    "DEZVALUIRI", "dezvaluiri", "Dezvaluiri",
    "DRAM(Ă|A)", "dram(ă|a)", "Dramă",
    "EXCLUSIV", "exclusiv", "Exclusiv",
    "EXCLUSIVITATE", "exclusivitate", "Exclusivitate",
    "EXPLOZIV", "exploziv", "Exploziv",
    "EXTRAORDINAR", "extraordinar", "Extraordinar",
    "FOTO", "foto", "Foto",
    "GALERIE", "galerie", "Galerie",
    "IMAGINI", "imagini", "Imagini",
    "INCENDIAR", "incendiar", "Incendiar",
    "INREGISTRARE", "inregistrare", "Inregistrare",
    "INTERVIU", "interviu", "Interviu",
    "LIVE", "live", "Live",
    "MISTER", "mister", "Mister",
    "PROVOCARE", "provocare", "Provocare",
    "REPLAY", "replay", "Replay",
    "REPORTAJ", "reportaj", "Reportaj",
    "REVELAȚIE", "revelație", "Revelație",
    "SCANDAL", "scandal", "Scandal",
    "SENZAȚIONAL", "senzațional", "Senzațional",
    "ȘOC", "șoc", "ŞOC", "Soc", "soc", "SOC",
    "TRAGEDIE", "tragedie", "Tragedie",
    "UIMITOR", "uimitor", "Uimitor",
    "ULTIM(Ă|A) OR(Ă|A)", "Ultim(ă|a) or(ă|a)", "ultima ora",
    "URGENT", "urgent", "Urgent",
    "VIDEO", "video", "Video"
]

PT = TypeVar('PT')
URL_REGEX = r"https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b([-a-zA-Z0-9()@:%_\+.~#?&//=]*)"
NUMBER_REGEX = r"\d+"
SPACE_REGEX = r"\s+"
SPECIAL_CHARS_REGEX = r"[&#.,?!\"\'‘’“”]|(\([^)]*\))"
BAIT_WORDS_REGEX = r'\b(' + '|'.join(BAIT_WORDS) + r')\b'
EMAIL_REGEX = r'\S*@\S*\s?'
HTML_REGEX = r'<[^<]+?>'


class TextPreprocessor(ABC, Generic[PT]):
    def __init__(self) -> None:
        pass

    @abstractmethod
    def __call__(self, dataset: DataFrame) -> PT:
        raise NotImplemented()


class AutoPreprocessor(TextPreprocessor[BatchEncoding]):
    def __init__(self, repo: str, cased: bool = True, max_length: int = 64) -> None:
        super().__init__()

        self.repo: str = repo
        self.cased = cased
        self.tokenizer = AutoTokenizer.from_pretrained(self.repo)
        self.max_length: int = max_length

    def __call__(self, dataset: DataFrame) -> Tuple[DataFrame, BatchEncoding]:
        dataset = dataset.drop(dataset[dataset['text'].str.contains('\t')].index)
        dataset = dataset.reset_index()

        dataset['text'] = dataset['text'].str.replace(
            URL_REGEX, '', regex=True)
        dataset['text'] = dataset['text'].str.replace(
            NUMBER_REGEX, '', regex=True)
        # dataset['text'] = dataset['text'].str.replace(
        #     SPECIAL_CHARS_REGEX, '', regex=True)
        # dataset['text'] = dataset['text'].str.replace(
        #     BAIT_WORDS_REGEX, '', regex=True)
        # dataset['text'] = dataset['text'].str.replace(
        #     EMAIL_REGEX, '', regex=True)
        dataset['text'] = dataset['text'].str.replace(
            HTML_REGEX, '', regex=True)
        dataset['text'] = dataset['text'].str.replace(
            SPACE_REGEX, ' ', regex=True)

        if not self.cased:
            dataset['text'] = dataset['text'].str.lower()

        # Manual preprocessing
        sentences: List[str] = []
        for text in tqdm(dataset['text'].tolist()):
            text: str = text \
                .replace("ţ", "ț") \
                .replace("ş", "ș") \
                .replace("Ţ", "Ț") \
                .replace("Ş", "Ș") \
                .replace('”', '"') \
                .replace('„', '"') \
                .replace('“', '"') \
                .replace('­', '') \
                .replace('–', '') \
                .replace('—', '')
            sentences.append(text)

        return dataset, self.tokenizer.__call__(
            sentences,
            add_special_tokens=True,
            padding='max_length',
            max_length=self.max_length,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )


class BERTPreprocessor(AutoPreprocessor):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(repo='dumitrescustefan/bert-base-romanian-cased-v1',
                         *args, **kwargs)


class MT5Preprocessor(AutoPreprocessor):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(repo='dumitrescustefan/mt5-base-romanian',
                         *args, **kwargs)


class RoBERTPreprocessor(AutoPreprocessor):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(repo='readerbench/RoBERT-base',
                         cased=False,
                         *args, **kwargs)



import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from typing import Dict
from transformers import BatchEncoding



class SatireDataset(Dataset):
    def __init__(self, dataset: pd.DataFrame, preprocessor: TextPreprocessor[BatchEncoding]):
        super().__init__()

        self.preprocessor: TextPreprocessor[BatchEncoding] = preprocessor
        self.dataset_processor, self.processed_text = self.preprocessor(dataset)

        self.classes = 'class' in self.dataset_processor.columns
        if self.classes:
            _, class_counts = np.unique(self.dataset_processor['class'], return_counts=True)
            class_counts = torch.from_numpy(class_counts)
            self.weights = 1 - class_counts / class_counts.sum()

    def __len__(self):
        return len(self.dataset_processor)

    def __getitem__(self, idx: int | slice) -> Dict[str, torch.Tensor]:
        input_ids, attention_mask = (self.processed_text['input_ids'][idx], self.processed_text['attention_mask'][idx])

        if self.classes:
            return dict(
                label=torch.tensor(self.dataset_processor['class'].iloc[idx]),
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        else:
            return dict(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

import torch
import torch.nn as nn
from torch import Tensor
from transformers import AutoModel, BertModel, MT5Model, MT5EncoderModel
from typing import Dict, List, Literal
from abc import ABC, abstractmethod
import numpy as np


class Args(object):
    def __init__(self) -> None:
        self.num_epochs = 3
        self.num_workers = 1
        self.batch_size = 64
        self.weight_decay = 5e-6
        self.learning_rate = 2e-5


class Output(object):
    def __init__(self, predictions: List[int], loss_seq: List[float], accy_seq: List[float],
                with_labels: bool) -> None:
        self.predictions_as_indx: np.ndarray = np.array(predictions)
        self.with_labels: bool = with_labels
        if self.with_labels:
            self.loss_seq: np.ndarray = np.array(loss_seq)
            self.accy_seq: np.ndarray = np.array(accy_seq)
            self.loss_mean: float = self.loss_seq.mean()
            self.accy_mean: float = self.accy_seq.mean()


class PretrainedFlatClassModel(ABC, nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

    @abstractmethod
    def create_layers(self) -> nn.Sequential:
        raise NotImplementedError()

    def unfreeze(self, layers: str | List[str] | List[nn.Module], unfreeze: bool = True) -> None:
        if isinstance(layers, list) and isinstance(layers[0], nn.Module):
            for module in layers:
                for param in module.parameters():
                    param.requires_grad = unfreeze
        elif layers == 'all':
            for param in self.parameters():
                param.requires_grad = unfreeze
        elif layers == 'none':
            self.requires_grad_(False)
            return
        elif isinstance(layers[0], str):
            for (param_name, param) in self.named_parameters():
                if param_name in layers:
                    param.requires_grad = unfreeze
        else:
            raise ValueError('invalid layers param - bad unfreeze')


class MT5FlatClassModel(PretrainedFlatClassModel):
    def __init__(self, unfreeze: Literal['all'] | Literal['none'] | List[str] = 'none', *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.fc_head = torch.nn.Linear(49152, 2)
        self.mt5: MT5Model = MT5EncoderModel.from_pretrained(
            "dumitrescustefan/mt5-base-romanian")
        self.unfreeze([
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[10],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.block[11],
            self.mt5.encoder.final_layer_norm
        ])

    def create_layers(self) -> nn.Sequential:
        pass

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        y = self.mt5(
            input_ids=x['input_ids'], attention_mask=x['attention_mask']).last_hidden_state
        y = y.view(y.shape[0], -1)
        y = self.fc_head(y)
        return y


class BERTFlatClassModel(PretrainedFlatClassModel):
    def __init__(self,
                 dropout: float = 0.1,
                 *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.layers = None
        self.dropout: float = dropout
        self.n_classes = 2
        self.repo = 'dumitrescustefan/bert-base-romanian-cased-v1'
        self.bert_model: BertModel = AutoModel.from_pretrained(self.repo)
        self.unfreeze([
            self.bert_model.pooler.dense,
            self.bert_model.encoder.layer[-1:]
        ])
        self.create_layers()

    def create_layers(self) -> None:
        self.layers = nn.Sequential(
            nn.Linear(in_features=768, out_features=512),
            nn.GELU(),
            nn.Linear(in_features=512, out_features=self.n_classes)
        )
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight)

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        input_ids: Tensor = x['input_ids']
        attention_mask: Tensor = x['attention_mask']
        _, output = self.bert_model(
            input_ids, attention_mask, return_dict=False)
        output = self.layers(output)

        return output


class RoBERTFlatClassModel(PretrainedFlatClassModel):
    def __init__(self,
                 dropout: float = 0.1,
                 *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        # Save internal params
        self.layers = None
        self.dropout: float = dropout

        # Init the base model
        self.n_classes = 2
        self.repo = 'readerbench/RoBERT-base'
        self.bert_model: BertModel = AutoModel.from_pretrained(self.repo)
        self.unfreeze([
            self.bert_model.pooler.dense,
            self.bert_model.encoder.layer[-1:]
        ])
        self.create_layers()

    def create_layers(self) -> None:
        self.layers = nn.Sequential(
            nn.Dropout(p=0.1),
            nn.Linear(in_features=768, out_features=self.n_classes)
        )

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        # Extract the relevant data
        input_ids: Tensor = x['input_ids']
        attention_mask: Tensor = x['attention_mask']

        # Call the pretrained model
        _, output = self.bert_model(
            input_ids, attention_mask, return_dict=False)

        # Add layers over it
        output = self.layers(output)

        return output


class Ensemble(nn.Module):
    def __init__(self, models: List[nn.Module], *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        # Init models
        self.models = models

    def forward(self, x: Dict[str, Tensor]) -> Tensor:
        y_hats = []

        for model in self.models:
            y_hat = model.forward(x).detach().cpu()
            y_hats.append(y_hat)

        stacked = torch.stack(y_hats, dim=1)
        predictions = torch.sum(stacked, dim=1)
        return predictions

import torch
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from typing import Dict, List
from sklearn.metrics import balanced_accuracy_score


def train(model: nn.Module, optimizer: Optimizer, loss_fn: nn.Module, train_loader: DataLoader, args: Args, device: torch.device):
    for epoch in range(args.num_epochs):
        model.train()

        epoch_loss: List[float] = []
        epoch_accuracy: List[float] = []
        predictions: List[int] = []
        for batch in tqdm(train_loader):
            batch: Dict[str, torch.Tensor] = {k: v.to(device) for k, v in batch.items()}

            y_true: torch.Tensor | np.ndarray = batch['label']
            y_pred: torch.Tensor | np.ndarray = model.forward(batch)

            optimizer.zero_grad()
            loss: torch.Tensor = loss_fn(y_pred, y_true)
            loss.backward()
            optimizer.step()

            y_true = y_true.detach().cpu().numpy()
            y_pred = y_pred.detach().argmax(dim=1).cpu().numpy()

            predictions.extend(y_pred.tolist())
            epoch_accuracy.append(balanced_accuracy_score(y_true, y_pred))
            epoch_loss.append(loss.detach().cpu().numpy())

            output = Output(predictions, epoch_loss, epoch_accuracy, with_labels=True)
            print('Train Epoch {} - Loss: {}, Balanced accuracy: {}'.format(epoch, output.loss_mean, output.accy_mean))

    return output


def evaluate(model: nn.Module, loss_fn: nn.Module, data_loader: DataLoader,
             with_labels: bool = True, device: torch.device = torch.device('cpu')) -> Output:
    model.eval()
    with torch.no_grad():
        epoch_loss: List[float] = []
        epoch_accy: List[float] = []
        predictions: List[int] = []

        for _, batch in enumerate(data_loader):
            batch: Dict[str, torch.Tensor] = {k: v.to(device) for k, v in batch.items()}

            y_pred: torch.Tensor | np.ndarray = model.forward(batch)
            if with_labels:
                y_true: torch.Tensor | np.ndarray = batch['label']
                loss: torch.Tensor = loss_fn(y_pred, y_true)

            y_pred = y_pred.detach().argmax(dim=1).cpu().numpy()
            predictions.extend(y_pred.tolist())

            if with_labels:
                y_true = y_true.detach().cpu().numpy()
                epoch_accy.append(balanced_accuracy_score(y_true, y_pred))
                epoch_loss.append(loss.detach().cpu().numpy())

    output = Output(predictions, epoch_loss, epoch_accy, with_labels=with_labels)
    return output

from typing import Tuple
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
import pathlib as pb
import pandas as pd


def read_data(data_dir: pb.Path, train_filename: str = 'train.csv', test_filename: str = 'test.csv') -> (
        Tuple[pd.DataFrame, pd.DataFrame]):
    train_data_raw = pd.read_csv(data_dir / train_filename, sep=',', index_col=None, encoding='utf-8')
    test_data_raw = pd.read_csv(data_dir / test_filename, sep=',', index_col=None, encoding='utf-8')
    train_data_raw['class'] = train_data_raw['class'].map({True: 1, False: 0})

    train_data_raw['text'] = train_data_raw['title'].fillna('') + ' ' + train_data_raw['content'].fillna('')
    test_data_raw['text'] = test_data_raw['title'].fillna('') + ' ' + test_data_raw['content'].fillna('')

    train_data_raw = train_data_raw.drop(columns='id', errors='ignore')
    test_data_raw = test_data_raw.drop(columns='id', errors='ignore')

    return train_data_raw, test_data_raw


def compute_metrics(labels, preds):
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    balanced_accuracy = (tp / (tp + fn) + specificity) / 2
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy,
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds),
        'specificity': specificity
    }

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random

from torch import optim
from pathlib import Path

from torch.utils.data import DataLoader

train_data_frame, test_data_frame \
    = read_data(Path('/kaggle/input/new-nitronlp2/'), 'train.csv', 'test.csv')

DEVICE: torch.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
SEED = 61

bert_preprocessor = BERTPreprocessor()
train_dataset = SatireDataset(train_data_frame, bert_preprocessor)
random.seed(SEED)
np.random.seed(SEED)
np.random.RandomState(SEED)
torch.manual_seed(SEED)

model_setup = [
    {
        'name': 'bert',
        'model_factory': lambda: BERTFlatClassModel().to(DEVICE),
        'optim_factory': lambda params: optim.AdamW(params, lr=Args().learning_rate, weight_decay=Args().weight_decay),
        'loss_fn': nn.CrossEntropyLoss(weight=train_dataset.weights.to(DEVICE)),
        'train_dataset': SatireDataset(train_data_frame, bert_preprocessor),
        'test_dataset': SatireDataset(test_data_frame, bert_preprocessor),
    },
    # {
    #     'name': 'mt5',
    #     'model_factory': lambda: MT5FlatClassModel().to(DEVICE),
    #     'optim_factory': lambda params: optim.AdamW(params, lr=Args().learning_rate, weight_decay=Args().weight_decay),
    #     'loss_fn': nn.CrossEntropyLoss(weight=train_dataset.weights),
    #     'train_dataset': SatireDataset(train_data_frame, MT5Preprocessor()),
    #     'test_dataset': SatireDataset(test_data_frame, MT5Preprocessor()),
    # },
    {
        'name': 'RoBERT',
        'model_factory': lambda: RoBERTFlatClassModel().to(DEVICE),
        'optim_factory': lambda params: optim.AdamW(params, lr=Args().learning_rate, weight_decay=Args().weight_decay),
        'loss_fn': nn.BCELoss(weight=train_dataset.weights),
        'train_dataset': SatireDataset(train_data_frame, RoBERTPreprocessor()),
        'test_dataset': SatireDataset(test_data_frame, RoBERTPreprocessor()),
    }
]

current_model_idx = 0
args = Args()
model = model_setup[current_model_idx]['model_factory']()
train_dataset = model_setup[current_model_idx]['train_dataset']
test_dataset = model_setup[current_model_idx]['test_dataset']
optimizer = model_setup[current_model_idx]['optim_factory'](model.parameters())
loss_fn = model_setup[current_model_idx]['loss_fn']

train_loader = DataLoader(train_dataset, args.batch_size, shuffle=True)

train_output: Output = train(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    train_loader=train_loader,
    args=args,
    device=DEVICE,
)
print('Last Epoch - Loss: {}, Accuracy: {}'.format(train_output.loss_mean, train_output.accy_mean))

test_loader: DataLoader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)
test_output: Output = evaluate(
    model=model,
    loss_fn=loss_fn,
    data_loader=test_loader,
    with_labels=False,
    device=DEVICE
)

output: pd.DataFrame = pd.DataFrame({'class': pd.Series(data=test_output.predictions_as_indx)})
output = output.reset_index()
output = output.rename(columns={'index': 'id'})
output.to_csv('submission.csv', index=False)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/397k [00:00<?, ?B/s]

100%|██████████| 36669/36669 [00:00<00:00, 140036.98it/s]


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

...

Train Epoch 2 - Loss: 0.008978238329291344, Balanced accuracy: 0.9972917060633435


100%|█████████▉| 1102/1103 [06:34<00:00,  2.78it/s]

Train Epoch 2 - Loss: 0.008970286697149277, Balanced accuracy: 0.997294163680346


100%|██████████| 1103/1103 [06:34<00:00,  2.80it/s]

Train Epoch 2 - Loss: 0.009008660912513733, Balanced accuracy: 0.9972642376156181
Last Epoch - Loss: 0.009008660912513733, Accuracy: 0.9972642376156181


In [5]:
!ls

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


m1.pth	submission.csv
